# RQ1 experiments.

This notebook investigates PR-BCDs behaviour with respect to block size and resampling epochs. It also investigates how and when different kinds of edges are useful to the PR-BCDs attack trajectory and final perturbation set.

Phases:
1. Train one victim model per seed.
2. Run small and large block size PR-BCD attacks to extract which edges are seen by each run.
3. Construct four groups of potentially useful and random edges.
4. Probe PR-BCD with edges from these four groups and investigate how they'd influence the PR-BCD loss at different checkpoints.
5. Inject edges from the four groups directly into PR-BCD at different checkpoints to investigate whether they influence the final PR-BCD attack loss.

In [1]:
# Setup: Import libraries and define directories.

from pathlib import Path
import os
import sys
import numpy as np
from IPython.display import display
import gc
import torch
import pandas as pd
from experiments import experiment_global_attack_direct
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(r"E:\Masterarbeit\AttackerGNN")
DATA_DIR = PROJECT_ROOT / "data"
CACHE_DIR = PROJECT_ROOT / "cache"

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments import experiment_train
from sparse_smoothing.utils import load_and_standardize

print("Project root:", PROJECT_ROOT)

Use from seml.experiment import setup_logger instead.
Note that seml.experiment.Experiment already includes the logger setup.
See https://github.com/TUM-DAML/seml/blob/master/examples/example_experiment.py


[08/23/26 12:31:20] WARNING  Importing setup_logger directly from seml is deprecated.                ]8;id=689808;file://E:\Anaconda\envs\Masterarbeit\Lib\site-packages\seml\__init__.py\__init__.py]8;;\:]8;id=387440;file://E:\Anaconda\envs\Masterarbeit\Lib\site-packages\seml\__init__.py#10\10]8;;\
                             Use from seml.experiment import setup_logger instead.                                 
                             Note that seml.experiment.Experiment already includes the logger setup.               
                             See                                                                                   
                             https://github.com/TUM-DAML/seml/blob/master/examples/example_experimen               
                             t.py                                                                                  

Project root: E:\Masterarbeit\AttackerGNN


## Dataset and victim model training

These cells define and train the victim models. All experiments in this thesis use the same victim model architecture and dataset; a separate victim instance is trained each seed. The RQ1 experiments are exclusively run on Cora-ML
The victim model is a 2-Layer GCN with PPR diffusion preprocessing adapted from the original PR-BCD implementation.


In [2]:
DATASET = "cora_ml"
SEEDS = [0]

MODEL_NAME = "GCN"
MODEL_LABEL = "GCN"
MODEL_STORAGE_TYPE = "demo_custom_split"

DROPOUT_VICTIM = 0.5
LR_VICTIM = 1e-2
WEIGHT_DECAY_VICTIM = 1e-3
PATIENCE_VICTIM = 300
MAX_EPOCHS_VICTIM = 3000

VICTIM_DEVICE = "cpu"
VICTIM_DATA_DEVICE = "cpu"

DATASET_PATH = DATA_DIR / f"{DATASET}.npz"

RQ1_DIR = PROJECT_ROOT / "extendedPlotting" / "final_runs" / "RQ1"
RQ1_DIR.mkdir(parents=True, exist_ok=True)

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset file not found: {DATASET_PATH}. "
        "Place the dataset in the repository data folder."
    )

CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Configured dataset={DATASET}, seeds={SEEDS}, model={MODEL_LABEL}")


Configured dataset=cora_ml, seeds=[0], model=GCN


In [3]:
# Train the victim model
for seed in SEEDS:

    print(f"Training or loading victim model for seed={seed}")

    stats = experiment_train.run(
        data_dir=str(DATA_DIR),
        dataset=DATASET,
        model_params=dict(
            label=MODEL_LABEL,
            model=MODEL_NAME,
            do_cache_adj_prep=True,
            n_filters=64,
            dropout=DROPOUT_VICTIM,
            svd_params=None,
            jaccard_params=None,
            gdc_params={"alpha": 0.15, "k": 64},
        ),
        train_params=dict(
            lr=LR_VICTIM,
            weight_decay=WEIGHT_DECAY_VICTIM,
            patience=PATIENCE_VICTIM,
            max_epochs=MAX_EPOCHS_VICTIM,
        ),
        binary_attr=False,
        make_undirected=True,
        seed=int(seed),
        artifact_dir=str(CACHE_DIR),
        model_storage_type=MODEL_STORAGE_TYPE,
        ppr_cache_params=dict(),
        device=VICTIM_DEVICE,
        data_device=VICTIM_DATA_DEVICE,
        display_steps=100,
        debug_level="info",
        custom_split_ratios=None,
    )

    print(
        f"seed={seed} | "
        f"clean accuracy={float(stats['accuracy']):.4f}"
    )

Training or loading victim model for seed=0


[08/23/26 12:31:29] INFO     {'dataset': 'cora_ml', 'model_params': {'label': 'GCN',        ]8;id=313618;file://E:\Masterarbeit\AttackerGNN\experiments\experiment_train.py\experiment_train.py]8;;\:]8;id=381645;file://E:\Masterarbeit\AttackerGNN\experiments\experiment_train.py#180\180]8;;\
                             'model': 'GCN', 'do_cache_adj_prep': True, 'n_filters': 64,                           
                             'dropout': 0.5, 'svd_params': None, 'jaccard_params': None,                           
                             'gdc_params': {'alpha': 0.15, 'k': 64}}, 'train_params':                              
                             {'lr': 0.01, 'weight_decay': 0.001, 'patience': 300,                                  
                             'max_epochs': 3000}, 'binary_attr': False, 'make_undirected':                         
                             True, 'seed': 0, 'artifact_dir':                                                      
                             'E:\\Masterarbeit\\AttackerGNN\\cache', 'model_storage_type':                         
                             'demo_custom_split', 'ppr_cache_params': {}, 'device': 'cpu',                         
                             'display_steps': 100, 'data_device': 'cpu'}                                           

                    INFO     Training set size: 140                                         ]8;id=22713;file://E:\Masterarbeit\AttackerGNN\experiments\experiment_train.py\experiment_train.py]8;;\:]8;id=341917;file://E:\Masterarbeit\AttackerGNN\experiments\experiment_train.py#216\216]8;;\

                    INFO     Validation set size: 140                                       ]8;id=63553;file://E:\Masterarbeit\AttackerGNN\experiments\experiment_train.py\experiment_train.py]8;;\:]8;id=997671;file://E:\Masterarbeit\AttackerGNN\experiments\experiment_train.py#217\217]8;;\

                    INFO     Test set size: 2530                                            ]8;id=1208;file://E:\Masterarbeit\AttackerGNN\experiments\experiment_train.py\experiment_train.py]8;;\:]8;id=971193;file://E:\Masterarbeit\AttackerGNN\experiments\experiment_train.py#218\218]8;;\

...

[08/23/26 12:31:44] INFO                                                                                ]8;id=235284;file://E:\Masterarbeit\AttackerGNN\rgnn_at_scale\train.py\train.py]8;;\:]8;id=430128;file://E:\Masterarbeit\AttackerGNN\rgnn_at_scale\train.py#81\81]8;;\
                             Epoch  600: loss_train: 0.06437, loss_val: 0.40359, acc_train: 1.00000,               
                             acc_val: 0.87857                                                                      

[08/23/26 12:31:46] INFO     Test accuracy is 0.8351778388023376 with seed 0                ]8;id=306750;file://E:\Masterarbeit\AttackerGNN\experiments\experiment_train.py\experiment_train.py]8;;\:]8;id=932347;file://E:\Masterarbeit\AttackerGNN\experiments\experiment_train.py#271\271]8;;\

                    INFO     Lock 2068019711440 acquired on                                         ]8;id=507704;file://E:\Anaconda\envs\Masterarbeit\Lib\site-packages\filelock.py\filelock.py]8;;\:]8;id=330599;file://E:\Anaconda\envs\Masterarbeit\Lib\site-packages\filelock.py#274\274]8;;\
                             E:\Masterarbeit\AttackerGNN\cache\demo_custom_split.json.lock                         

                    INFO     Lock 2068019711440 released on                                         ]8;id=246159;file://E:\Anaconda\envs\Masterarbeit\Lib\site-packages\filelock.py\filelock.py]8;;\:]8;id=600261;file://E:\Anaconda\envs\Masterarbeit\Lib\site-packages\filelock.py#318\318]8;;\
                             E:\Masterarbeit\AttackerGNN\cache\demo_custom_split.json.lock                         

seed=0 | clean accuracy=0.8352


## Experimental configuration

Defines:
1. The used small and large block sizes used by PR-BCD as injection and probe targets and reference for group construction respectively.
2. Shared PR-BCD budget fraction, epochs and finetuning epochs.
3. Shared probe and injection checkpoints.
4. Four probe and injection groups.
5. Number of edges used to fill these four groups.
6. Shared sampling seed.

In [4]:

# Experiment configuration

MISS_EPSILON = 0.015 # PR-BCD budget fraction.

SMALL_BLOCK_SIZES = [250, 500, 1_000, 2_000, 3_000, 4_000] # Small block sizes.
LARGE_BLOCK_SIZE = 50_000 # Large PR-BCD reference block size.

EPOCHS = 300 # Shared total number of epochs
FINE_TUNE_EPOCHS = 250 # Shared number of finetuning epochs.
WITH_EARLY_STOPPING = False

# Groups and group labels.
GROUPS = ["large_final_missed", "large_top_weight", "large_only_random", "random_unseen_control"]
GROUP_LABELS = {
    "large_final_missed": "large final missed",
    "large_top_weight": "large top weight",
    "large_only_random": "large only random",
    "random_unseen_control": "unseen control",
}

CHECKPOINTS = [0, 24, 49, 99, 299] # Checkpoints

# Number of edges from the groups used for probe/injection experiments
N_TOP_WEIGHT = 100
N_RANDOM_LARGE_ONLY = 100
N_RANDOM_UNSEEN = 100

# Batch size of the injection experiment
N_INJECTION_EDGES = 10

SAMPLING_SEEDS = {seed: 730_000 + seed * 10_000 for seed in SEEDS}

ARTIFACT_DIR = str(PROJECT_ROOT / "cache")
PERT_ADJ_STORAGE_TYPE = "evasion_global_adj"
PERT_ATTR_STORAGE_TYPE = "evasion_global_attr"

In [5]:
graph_sparse = load_and_standardize(str(DATA_DIR / f"{DATASET}.npz"))

N_NODES = int(graph_sparse.adj_matrix.shape[0])
N_UNDIRECTED = int(graph_sparse.adj_matrix.nnz // 2)
N_POSSIBLE_EDGES = N_NODES * (N_NODES - 1) // 2

ATTACK_BUDGET = max(1, round(MISS_EPSILON * N_UNDIRECTED)) # Converts budget fraction into actual PR-BCD budget.
ALL_BLOCK_SIZES = sorted(set(SMALL_BLOCK_SIZES + [LARGE_BLOCK_SIZE]))

# Check from PR-BCD. PR-BCD cant run when block size is smaller than requested number of perturbations.
invalid_block_sizes = [
    block_size
    for block_size in ALL_BLOCK_SIZES
    if block_size <= ATTACK_BUDGET
]

if invalid_block_sizes:
    raise ValueError(
        f"All block sizes must exceed "
        f"attack budget={ATTACK_BUDGET}. "
        f"Invalid block sizes: {invalid_block_sizes}."
    )

print("Graph nodes:", N_NODES)
print("Undirected edges:", N_UNDIRECTED)
print("Possible edge flips:", f"{N_POSSIBLE_EDGES:,}")
print("Attack budget:", ATTACK_BUDGET)

Graph nodes: 2810
Undirected edges: 7981
Possible edge flips: 3,946,645
Attack budget: 120


## Phase 1 — PR-BCD discovery runs.

This cell runs PR-BCD on different block sizes and infers which edges are ever seen by the different runs. These discovere edges are then used to construct the four groups.

In [ ]:

# Helper: extracts all edges that a given run has discovered.
def get_ever_seen_edges(result):
    epoch_blocks = result["attack_statistics"]["block_diagnostics"]["epoch_blocks"]
    return torch.unique(torch.cat([torch.as_tensor(block).cpu().long()for block in epoch_blocks.values()]),sorted=True)

# Helper: Runs PR-BCD with block diagnostics enabled.
def run_prbcd(victim_seed, block_size, sampling_seed):

    attack_params = {
        "block_size": block_size,
        "epochs": EPOCHS,
        "fine_tune_epochs": FINE_TUNE_EPOCHS,
        "with_early_stopping": WITH_EARLY_STOPPING,
        "keep_heuristic": "WeightOnly",
        "do_synchronize": True,
        "loss_type": "tanhMargin",
        "block_diagnostics_enabled": True,
        "attack_sampling_seed": sampling_seed,
    }

    return experiment_global_attack_direct.run(
        graph=graph_sparse,
        data_dir=str(PROJECT_ROOT / "data"),
        dataset=DATASET,
        attack="PRBCD",
        attack_params=attack_params,
        selector_params={},
        epsilons=[MISS_EPSILON],
        binary_attr=False,
        make_undirected=True,
        seed=int(victim_seed),
        artifact_dir=ARTIFACT_DIR,
        pert_adj_storage_type=PERT_ADJ_STORAGE_TYPE,
        pert_attr_storage_type=PERT_ATTR_STORAGE_TYPE,
        model_label=MODEL_LABEL,
        model_storage_type=MODEL_STORAGE_TYPE,
        device="cpu",
        data_device="cpu",
        debug_level="info",
        semi=True,
        use_cert="none",
    )

# Run loop, runs PR-BCD for all block sizes and victim models.
discovery_runs = []
for victim_seed in SEEDS:
    for block_size in ALL_BLOCK_SIZES:
        result = run_prbcd(
            victim_seed=victim_seed,
            block_size=block_size,
            sampling_seed=SAMPLING_SEEDS[victim_seed],
        )
        diagnostics = (result["attack_statistics"]["block_diagnostics"])
        ever_seen = get_ever_seen_edges(result)
        max_weight = diagnostics["max_weight"][ever_seen].cpu().float()
        final_edges = torch.as_tensor(diagnostics["final_linear_ids"]).cpu().long()
        final_accuracy = torch.as_tensor(result["results"][0]["accuracy"]).item()
        discovery_runs.append({
            "seed": victim_seed,
            "sampling_seed": SAMPLING_SEEDS[victim_seed],
            "block_size": block_size,
            "ever_seen": ever_seen,
            "max_weight": max_weight,
            "final_edges": final_edges,
            "final_accuracy": final_accuracy,
        })

        del result
        gc.collect()

# Small overview table
discovery_df = pd.DataFrame([
    {
        "seed": run["seed"],
        "sampling_seed": run["sampling_seed"],
        "block_size": run["block_size"],
        "n_ever_seen": run["ever_seen"].numel(),
        "n_final_edges": run["final_edges"].numel(),
        "final_accuracy": run["final_accuracy"],
    }
    for run in discovery_runs
])

display(discovery_df)

## Phase 2 — Construction of the four groups used in subsequent probe and injection experiments.

The four groups used in the thesis are:

- `large_final_missed`: selected by the large PR-BCD and never seen by the small.
- `large_top_weight`: large PR-BCD discovered edges with the largest assigned edge weight that the small run missed.
- `large_only_random`: random edges seen only by the large run.
- `random_unseen_control`: uniform edges seen by neither paired run.

In [9]:

candidate_rows = []
for victim_seed in SEEDS:
    large_run = next(run for run in discovery_runs
        if run["seed"] == victim_seed
        and run["block_size"] == LARGE_BLOCK_SIZE
    )
    large_seen = large_run["ever_seen"]
    large_final = large_run["final_edges"]
    large_max_weight = large_run["max_weight"]

    for small_block_size in SMALL_BLOCK_SIZES:

        small_run = next(run for run in discovery_runs
            if run["seed"] == victim_seed
            and run["block_size"] == small_block_size
        )
        small_seen = small_run["ever_seen"]

        # Edges seen by large but not by small
        large_only_mask = ~torch.isin(large_seen,small_seen)
        large_only_edges = large_seen[large_only_mask]
        large_only_weights = large_max_weight[large_only_mask]

        # Group 1: large_final_missed
        large_final_missed = large_final[~torch.isin(large_final, small_seen)]

        # Group 2: large_top_signal
        n_top = min(N_TOP_WEIGHT,large_only_edges.numel())
        top_indices = torch.topk(large_only_weights,k=n_top).indices
        large_top_weight = large_only_edges[top_indices]

        # Group 3: large_only_random
        rng = np.random.default_rng(900_000 + victim_seed * 10_000 + small_block_size)
        n_random = min(N_RANDOM_LARGE_ONLY, large_only_edges.numel())
        random_indices = rng.choice(
            large_only_edges.numel(),
            size=n_random,
            replace=False,
        )
        large_only_random = large_only_edges[torch.as_tensor(random_indices)]

        # Group 4: random edges unseen by both runs
        all_edges = torch.arange(N_POSSIBLE_EDGES)
        seen_by_either = torch.unique(torch.cat([small_seen, large_seen]))
        unseen_edges = all_edges[~torch.isin(all_edges, seen_by_either)]
        random_indices = torch.randperm(unseen_edges.numel())[:N_RANDOM_UNSEEN]
        random_unseen = unseen_edges[random_indices]

        groups = {
            "large_final_missed": large_final_missed,
            "large_top_weight": large_top_weight,
            "large_only_random": large_only_random,
            "random_unseen_control": random_unseen,
        }

        for group, edges in groups.items():
            for edge_id in edges.tolist():
                candidate_rows.append({
                    "seed": victim_seed,
                    "small_block_size": small_block_size,
                    "group": group,
                    "linear_id": edge_id,
                })


candidate_df = pd.DataFrame(candidate_rows)

## Phase 3: Probe experiments

This cell runs the PR-BCD small run probe experiments. Edges from the four groups are provided to all small block size PR-BCD attacks and one by one replace the lowest weighted edge in the PR-BCD block. Then a full simulated PR-BCD gradient update and projection step is performed, so the inserted edge has the chance to accumulate sufficient edge weight. The loss is recorded and the effect on the loss of the probe is measured $\Delta\mathcal{L}_{\mathrm{one-step}}$.

In [ ]:

# Helper: run PR-BCD with probed edges provided.
def run_probe_prbcd(victim_seed, block_size, sampling_seed, probe_ids, probe_groups):

    attack_params = {
        "block_size": block_size,
        "epochs": EPOCHS,
        "fine_tune_epochs": FINE_TUNE_EPOCHS,
        "with_early_stopping": WITH_EARLY_STOPPING,
        "keep_heuristic": "WeightOnly",
        "do_synchronize": True,
        "loss_type": "tanhMargin",
        "block_diagnostics_enabled": True,
        "attack_sampling_seed": sampling_seed,
        "probe_ids": probe_ids, # Probe edge ids.
        "probe_groups": probe_groups, # Probe group indicator.
        "probe_checkpoint_epochs": CHECKPOINTS, # Probing checkpoints.
    }

    return experiment_global_attack_direct.run(
        graph=graph_sparse,
        data_dir=str(PROJECT_ROOT / "data"),
        dataset=DATASET,
        attack="PRBCD",
        attack_params=attack_params,
        selector_params={},
        epsilons=[MISS_EPSILON],
        binary_attr=False,
        make_undirected=True,
        seed=int(victim_seed),
        artifact_dir=ARTIFACT_DIR,
        pert_adj_storage_type=PERT_ADJ_STORAGE_TYPE,
        pert_attr_storage_type=PERT_ATTR_STORAGE_TYPE,
        model_label=MODEL_LABEL,
        model_storage_type=MODEL_STORAGE_TYPE,
        device="cpu",
        data_device="cpu",
        debug_level="info",
        semi=True,
        use_cert="none",
    )

# Probe loop. Probes for all small block sizes and trained victim models.
probe_rows = []
for victim_seed in SEEDS:
    for small_block_size in SMALL_BLOCK_SIZES:
        candidates = candidate_df[(candidate_df["seed"] == victim_seed) & (candidate_df["small_block_size"] == small_block_size)]

        probe_ids = candidates["linear_id"].astype(int).tolist()
        probe_groups = candidates["group"].tolist()

        result = run_probe_prbcd(
            victim_seed=victim_seed,
            block_size=small_block_size,
            sampling_seed=SAMPLING_SEEDS[victim_seed],
            probe_ids=probe_ids,
            probe_groups=probe_groups,
        )

        for probe in result["attack_statistics"]["probe_results"]:
            probe_rows.append({
                "seed": victim_seed,
                "small_block_size":small_block_size,
                **probe,
            })

        del result
        gc.collect()


probe_df = pd.DataFrame(probe_rows)
print("Total probes:", len(probe_df))
probe_df.to_csv(RQ1_DIR / "probe_results.csv", index=False)
display(probe_df)

### Probe bloxplot.

The probe boxplot shows the spread of all replacement loss effects over probe epochs and trained victim models.

- **x-axis:** Candidate group $\mathcal{G}$.
- **y-axis:** One-step PRBCD replacement loss effect $\Delta\mathcal{L}_{\mathrm{one-step}}$.

Plot is shown in Figure 4.


In [ ]:

# Dataframe for plots. Only retains the replacement loss effect.
plot_data = [
    probe_df.loc[
        probe_df["group"] == group,
        "delta_loss",
    ].values
    for group in GROUPS
]

plt.figure(figsize=(10, 5))
plt.boxplot(plot_data, labels=GROUPS, showfliers=False,)
plt.axhline(0, linestyle="--")
plt.title("Replacement loss effect of missed edges")
plt.ylabel(r"One-step PRBCD replacement loss effect")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(
    RQ1_DIR / "boxplot_replacement_loss.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

### Probe-Checkpoint plots.

These plots show the replacement loss effect $\Delta L_{\mathrm{one-step}}$ over checkpoints $t^*$ and for each block size. It is constructed to infer at which epoch the replacement loss effect had the greatest magnitute. It also infers whether the magnitude of the replacement loss effect changes with changes in block size.

- **x-axis:** PRBCD checkpoint epoch $t^*$ at which the candidate edges are probed.
- **y-axis:** Mean replacement loss effect $\Delta L_{\mathrm{one-step}}$ at the respective checkpoints.
- **lines:** Separate the effects of the four groups.

Plots are shown in Figures 5, 14 and 15.


In [ ]:

# Mean over all seeds and all candidate edges within each group
time_df = (probe_df.groupby([
            "small_block_size",
            "checkpoint_epoch",
            "group"],
        as_index=False,
    )["delta_loss"].mean())

for block_size in SMALL_BLOCK_SIZES:
    block_df = time_df[time_df["small_block_size"] == block_size]
    plt.figure(figsize=(10, 5))
    for group in GROUPS:
        group_df = block_df[block_df["group"] == group]
        plt.plot(
            group_df["checkpoint_epoch"],
            group_df["delta_loss"],
            marker="o",
            label=group,
        )

    plt.axhline(0, linestyle="--")

    plt.title(
        f"Missed-edge usefulness over PR-BCD epochs — B={block_size}"
    )

    plt.xlabel(r"Checkpoint epoch $t^*$")
    plt.ylabel("Mean one-step PRBCD replacement loss effect")

    plt.legend()
    plt.tight_layout()

    plt.savefig(
        RQ1_DIR / f"replacement_loss_checkpoint_B{block_size}.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

### Probe positive replacement loss effect fraction plot.

Shows fraction of edges from which group had a positive effect on replacement loss effect.

- **x-axis:** Group from which the edge was probed from
- **y-axis:** Fraction of probed edges with a positive replacement loss effect $\Delta L > 0$. A higher value means that a larger share of edges in the respective group increased the tanh margin loss of PR-BCD.

Plot shown in Figure 6.


In [ ]:

positive_df = probe_df.copy()
positive_df["positive_effect"] = positive_df["delta_loss"] > 0

# Fraction positive within each seed/checkpoint/group
positive_by_checkpoint = (positive_df.groupby([
            "seed",
            "small_block_size",
            "checkpoint_epoch",
            "group"], as_index=False,
    )["positive_effect"].mean()
)

# Average those fractions over seeds and checkpoints
positive_summary = (positive_by_checkpoint
    .groupby([
            "small_block_size",
            "group"], as_index=False)["positive_effect"]
    .mean())

plt.figure(figsize=(10, 5))
for block_size in SMALL_BLOCK_SIZES:
    block_df = positive_summary[positive_summary["small_block_size"] == block_size]
    values = [block_df.loc[block_df["group"] == group, "positive_effect"].iloc[0]
        for group in GROUPS
    ]

    plt.plot(GROUPS, values, marker="o", label=f"B={block_size:,}",)

plt.title("How often missed candidates improve the PRBCD loss")
plt.ylabel("Mean fraction of probes with positive loss effect")

plt.xticks(rotation=20, ha="right")
plt.ylim(0, 1)
plt.legend()

plt.tight_layout()

plt.savefig(
    RQ1_DIR / "replacement_loss_positive.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## Phase 4: Injection experiment.

For each small run, each trained victim model and each group the experiment provides batches of edges to the small block size PR-BCD that then replaces the lowest weighted edges in the current PR-BCD state. After that, the PR-BCD continues as in the standard setting defined in the thesis. The goal is to decern whether, when and edges from which group have an effect on the final accuracy of PR-BCD when injected.

In [ ]:

# Helper: Runs PR-BCD with edge injections.
def run_injection_prbcd(
    victim_seed,
    block_size,
    sampling_seed,
    injection_ids=None,
    injection_epoch=None,
):

    attack_params = {
        "block_size": block_size,
        "epochs": EPOCHS,
        "fine_tune_epochs": FINE_TUNE_EPOCHS,
        "with_early_stopping": WITH_EARLY_STOPPING,
        "keep_heuristic": "WeightOnly",
        "do_synchronize": True,
        "loss_type": "tanhMargin",
        "attack_sampling_seed": sampling_seed,
        "injection_ids": injection_ids or [],
        "injection_epoch": injection_epoch,
    }

    return experiment_global_attack_direct.run(
        graph=graph_sparse,
        data_dir=str(PROJECT_ROOT / "data"),
        dataset=DATASET,
        attack="PRBCD",
        attack_params=attack_params,
        selector_params={},
        epsilons=[MISS_EPSILON],
        binary_attr=False,
        make_undirected=True,
        seed=int(victim_seed),
        artifact_dir=ARTIFACT_DIR,
        pert_adj_storage_type=PERT_ADJ_STORAGE_TYPE,
        pert_attr_storage_type=PERT_ATTR_STORAGE_TYPE,
        model_label=MODEL_LABEL,
        model_storage_type=MODEL_STORAGE_TYPE,
        device="cpu",
        data_device="cpu",
        debug_level="info",
        semi=True,
        use_cert="none",
    )

# Injection loop.
injection_rows = []

for victim_seed in SEEDS:
    for small_block_size in SMALL_BLOCK_SIZES:

        baseline_row = discovery_df[
            (discovery_df["seed"] == victim_seed)
            & (discovery_df["block_size"] == small_block_size)
            & (discovery_df["sampling_seed"] == SAMPLING_SEEDS[victim_seed])]

        baseline_final_accuracy = baseline_row.iloc[0]["final_accuracy"]

        # Loop over checkpoints and groups.
        for injection_epoch in CHECKPOINTS:
            for group in GROUPS:

                candidates = candidate_df[
                    (candidate_df["seed"] == victim_seed)
                    & (candidate_df["small_block_size"] == small_block_size)
                    & (candidate_df["group"] == group)]

                # Select exactly 10 candidates from the group.
                injection_ids = (candidates["linear_id"].head(N_INJECTION_EDGES).tolist())

                # Run injection PR-BCD
                result = run_injection_prbcd(
                    victim_seed=victim_seed,
                    block_size=small_block_size,
                    sampling_seed=SAMPLING_SEEDS[victim_seed],
                    injection_ids=injection_ids,
                    injection_epoch=injection_epoch,
                )

                injection_final_accuracy = float(result["results"][0]["accuracy"])

                injection_rows.append({
                    "seed": victim_seed,
                    "small_block_size": small_block_size,
                    "injection_epoch": injection_epoch,
                    "injection_update": injection_epoch + 1,
                    "group": group,
                    "n_injected": len(injection_ids),
                    "baseline_final_accuracy": baseline_final_accuracy,
                    "injection_final_accuracy": injection_final_accuracy,
                    "final_accuracy_delta":
                        baseline_final_accuracy
                        - injection_final_accuracy,
                })

                del result
                gc.collect()

injection_df = pd.DataFrame(injection_rows)


 40%|████      | 120/300 [01:48<02:42,  1.11it/s]

[08/23/26 15:00:25] INFO                                                                               ]8;id=696102;file://E:\Masterarbeit\AttackerGNN\rgnn_at_scale\attacks\prbcd.py\prbcd.py]8;;\:]8;id=781706;file://E:\Masterarbeit\AttackerGNN\rgnn_at_scale\attacks\prbcd.py#269\269]8;;\
                             Epoch: 120 Loss: -0.614463746547699 Accuracy: 81.107 %                                
                                                                                                                   


 47%|████▋     | 140/300 [02:06<02:22,  1.12it/s]

[08/23/26 15:00:43] INFO                                                                               ]8;id=786146;file://E:\Masterarbeit\AttackerGNN\rgnn_at_scale\attacks\prbcd.py\prbcd.py]8;;\:]8;id=554042;file://E:\Masterarbeit\AttackerGNN\rgnn_at_scale\attacks\prbcd.py#269\269]8;;\
                             Epoch: 140 Loss: -0.6134287118911743 Accuracy: 80.988 %                               
                                                                                                                   


 53%|█████▎    | 160/300 [02:24<02:02,  1.14it/s]

...

Standard PRBCD -> random initial block


                    INFO                                                                               ]8;id=386869;file://E:\Masterarbeit\AttackerGNN\rgnn_at_scale\attacks\prbcd.py\prbcd.py]8;;\:]8;id=220478;file://E:\Masterarbeit\AttackerGNN\rgnn_at_scale\attacks\prbcd.py#211\211]8;;\
                             Before the attack - Loss: -0.652926504611969 Accuracy: 83.518 %                       
                                                                                                                   


  0%|          | 0/300 [00:00<?, ?it/s]

[08/23/26 15:08:14] INFO                                                                               ]8;id=271558;file://E:\Masterarbeit\AttackerGNN\rgnn_at_scale\attacks\prbcd.py\prbcd.py]8;;\:]8;id=565270;file://E:\Masterarbeit\AttackerGNN\rgnn_at_scale\attacks\prbcd.py#269\269]8;;\
                             Epoch: 0 Loss: -0.6529242396354675 Accuracy: 83.399 %                                 
                                                                                                                   


  6%|▌         | 17/300 [00:16<04:44,  1.00s/it]

### Injection plots

Each point represents one injected batch for one victim seed. Boxes summarize these batch effects for a fixed block size, injection checkpoint, and candidate group.

- **x-axis:** Injection checkpoint $t^*$ and candidate group $\mathcal{G}$.
- **y-axis:** Reduction in final attacked accuracy relative to PR-BCD without injection in PP.

Plots shown in Figures 7, 16 and 17

In [ ]:

Path(
    r"E:\Masterarbeit\AttackerGNN\extendedPlotting\final_runs\RQ1"
).mkdir(parents=True, exist_ok=True)

for block_size in SMALL_BLOCK_SIZES:

    box_data = []
    box_labels = []
    box_positions = []

    position = 1.0

    for injection_epoch in [epoch for epoch in CHECKPOINTS[:-2]]:
        for group in GROUPS:

            # Average multiple injection batches per seed first,
            # so each seed contributes one observation.
            values = (
                injection_df.loc[
                    (injection_df["small_block_size"] == block_size)
                    & (injection_df["injection_epoch"] == injection_epoch)
                    & (injection_df["group"] == group)
                ]
                .groupby("seed")["final_accuracy_delta"]
                .mean()
                .dropna()
                .to_numpy()
            )

            box_data.append(values)

            box_labels.append(
                f"{injection_epoch}\n"
                + group
                .replace("large_final_missed", "final missed")
                .replace("large_top_signal", "top signal")
                .replace("large_only_random", "large random")
                .replace("random_unseen_control", "unseen control")
            )

            box_positions.append(position)
            position += 1.0

        # Extra spacing between injection epochs
        position += 0.7

    plt.figure(figsize=(max(12, 0.8 * len(box_data)), 6))
    plt.boxplot(
        box_data,
        positions=box_positions,
        tick_labels=box_labels,
        showfliers=False,
        widths=0.65,
    )

    # Individual seed observations
    for pos, values in zip(
        box_positions,
        box_data,
    ):
        plt.scatter(
            np.full(
                values.shape,
                pos,
                dtype=float,
            ),
            values,
            s=24,
            alpha=0.7,
        )

    plt.axhline(0,linestyle="--")
    plt.xlabel("Injection epoch and candidate group")

    plt.ylabel(
        "Final accuracy reduction vs. baseline\n"
        "(positive = stronger attack)"
    )

    plt.title(f"Injection effect of candidate groups — B={block_size:,}")
    plt.xticks(rotation=40, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        RQ1_DIR / f"injection_effect_B{block_size}.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()
    plt.close()
